# Triplet permutation entropy

This notebook groups all 6 permutations of the same triplet together by sorting the three image IDs. For each canonical triplet `(i, j, k)`, it counts how often each item is the odd one out, converts those counts to percentages, then computes raw Shannon entropy and the normalized agreement score:

`agreement_score = 1 + sum(p_m * log(p_m)) / log(3)`

Rows in `trainset.txt` are expected to be `chosen_a chosen_b odd_one_out`, so the third column is the odd item.

In [5]:
from collections import defaultdict
import csv
import math
from pathlib import Path

input_path = Path("data/testset1.txt")
output_path = Path("output/testset1_triplet_entropy.csv")

# counts[(i, j, k)] = [odd_i_count, odd_j_count, odd_k_count]
# where (i, j, k) is the sorted/canonical triplet.
counts = defaultdict(lambda: [0, 0, 0])
row_count = 0

with input_path.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 3:
            raise ValueError(f"Line {line_number} has {len(values)} fields, expected 3")

        chosen_a, chosen_b, odd_one_out = map(int, values)
        triplet = tuple(sorted((chosen_a, chosen_b, odd_one_out)))
        odd_position = triplet.index(odd_one_out)
        counts[triplet][odd_position] += 1
        row_count += 1

def shannon_entropy(probabilities):
    """Raw Shannon entropy using natural log, in nats."""
    return -sum(p * math.log(p) for p in probabilities if p > 0)

with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "i", "j", "k", "total_count",
        "odd_i_count", "odd_j_count", "odd_k_count",
        "odd_i_pct", "odd_j_pct", "odd_k_pct",
        "shannon_entropy", "agreement_score",
    ])

    for triplet in sorted(counts):
        odd_counts = counts[triplet]
        total = sum(odd_counts)
        probabilities = [count / total for count in odd_counts]
        entropy = shannon_entropy(probabilities)
        if abs(entropy) < 1e-15:
            entropy = 0.0
        agreement = 1 - entropy / math.log(3)
        agreement = min(1.0, max(0.0, agreement))

        writer.writerow([
            *triplet,
            total,
            *odd_counts,
            *(100 * p for p in probabilities),
            entropy,
            agreement,
        ])

print(f"Read {row_count:,} rows from {input_path}")
print(f"Found {len(counts):,} unique canonical triplets")
print(f"Wrote {output_path}")

Read 15,640 rows from data/testset1.txt
Found 1,000 unique canonical triplets
Wrote output/testset1_triplet_entropy.csv
